In [1]:
import pandas as pd

ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")
df = pd.merge(ratings, movies, on="movieId")
df = df[['userId', 'movieId', 'rating', 'title', 'genres']]

print(df.head().to_string(index=False))

 userId  movieId  rating                       title                                      genres
      1        1     4.0            Toy Story (1995) Adventure|Animation|Children|Comedy|Fantasy
      1        3     4.0     Grumpier Old Men (1995)                              Comedy|Romance
      1        6     4.0                 Heat (1995)                       Action|Crime|Thriller
      1       47     5.0 Seven (a.k.a. Se7en) (1995)                            Mystery|Thriller
      1       50     5.0  Usual Suspects, The (1995)                      Crime|Mystery|Thriller


In [2]:
user_movie_matrix = df.pivot_table(
    index='userId',
    columns='title',
    values='rating'
).fillna(0)

user_movie_matrix.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
genres = df['genres'].str.split('|').explode()

genres.value_counts().head(10)

genres
Drama        41928
Comedy       39053
Action       30635
Thriller     26452
Adventure    24161
Romance      18124
Sci-Fi       17243
Crime        16681
Fantasy      11834
Children      9208
Name: count, dtype: int64

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


In [13]:
def get_top_movies(user_id, n=18):
    avg = df.groupby('title')['rating'].mean()
    count = df.groupby('title')['rating'].count()

    top = pd.DataFrame({'avg': avg, 'count': count})
    top = top[top['count'] > 50]
    top = top.sort_values(by='avg', ascending=False)

    user_movies = user_movie_matrix.loc[user_id]
    watched = user_movies[user_movies > 0].index

    top = top[~top.index.isin(watched)]

    return top.head(n).index.tolist()

In [14]:
def recommend_by_genre(user_id, movie_name, n=18):
    temp = df.copy()

    genre = temp[temp['title'] == movie_name]['genres'].values
    if len(genre) == 0:
        return []

    genre_set = set(genre[0].split('|'))

    temp['score'] = temp['genres'].apply(
        lambda x: len(genre_set.intersection(set(x.split('|'))))
    )

    recs = temp[(temp['score'] >= 2) & (temp['title'] != movie_name)]

    user_movies = user_movie_matrix.loc[user_id]
    watched = user_movies[user_movies > 0].index

    recs = recs[~recs['title'].isin(watched)]

    return recs.sort_values(by=['score', 'rating'], ascending=False)['title'].drop_duplicates().head(n).tolist()

In [15]:
def recommend_users(user_id, n=18):
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]

    user_movies = user_movie_matrix.loc[user_id]
    watched = user_movies[user_movies > 0].index

    scores = {}

    for sim_user, sim_score in similar_users.items():
        sim_movies = user_movie_matrix.loc[sim_user]

        for movie, rating in sim_movies.items():
            if movie not in watched and rating > 0:
                scores[movie] = scores.get(movie, 0) + sim_score * rating

    sorted_movies = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [m[0] for m in sorted_movies[:n]]


In [16]:
def get_movies_by_genre(user_id, genre, n=18):
    g = df[df['genres'].str.contains(genre, case=False)]

    user_movies = user_movie_matrix.loc[user_id]
    watched = user_movies[user_movies > 0].index

    g = g[~g['title'].isin(watched)]

    top = g.groupby('title')['rating'].mean().sort_values(ascending=False)

    return top.head(n).index.tolist()

In [17]:
def search_movies(user_id, name, n=12):
    result = df[df['title'].str.contains(name, case=False)]

    user_movies = user_movie_matrix.loc[user_id]
    watched = user_movies[user_movies > 0].index

    result = result[~result['title'].isin(watched)]

    return result['title'].drop_duplicates().head(n).tolist()


# ---------------- MAIN ---------------- #

user_id = int(input("Enter User ID: "))

# safety check
if user_id not in user_movie_matrix.index:
    print("❌ User not found")
else:
    query = input("🔍 Search Movie: ")

    if query == "":
        print("🔥 Top Movies:")
        print(get_top_movies(user_id))

    else:
        results = search_movies(user_id, query)

        print("\n🔍 Search Results:")
        for m in results:
            print("🎬", m)

🔥 Top Movies:
['Godfather, The (1972)', 'Fight Club (1999)', 'Cool Hand Luke (1967)', 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)', 'Rear Window (1954)', 'Godfather: Part II, The (1974)', 'Goodfellas (1990)', 'Casablanca (1942)', 'Usual Suspects, The (1995)', 'Princess Bride, The (1987)', 'Star Wars: Episode IV - A New Hope (1977)', "Schindler's List (1993)", 'Apocalypse Now (1979)', 'American History X (1998)', 'Star Wars: Episode V - The Empire Strikes Back (1980)', 'Chinatown (1974)', 'Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)', "One Flew Over the Cuckoo's Nest (1975)"]
